In [1]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number
import os
from PIL import Image
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import random
from collections import defaultdict, Counter
from glob import glob
import kagglehub
from pyspark.sql.functions import rand, col, udf
import matplotlib.pyplot as plt

In [2]:
path = kagglehub.dataset_download("daniildeltsov/traffic-signs-gtsrb-plus-162-custom-classes")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\jaron\.cache\kagglehub\datasets\daniildeltsov\traffic-signs-gtsrb-plus-162-custom-classes\versions\1


In [3]:
# Check the current working directory
current_directory = os.getcwd()
print(f"Current working directory: {current_directory}")

# List the files in the current working directory
files_in_directory = os.listdir(current_directory)
print(f"Files in current directory: {files_in_directory}")

Current working directory: C:\Users\jaron\Downloads
Files in current directory: ['.ipynb_checkpoints', '449338860_8085469114837938_2253952122707354560_n.jpg', '449795110_911576291175200_3938656764344245407_n.jpg', '449974480_8750771461618283_7323365492380633485_n.jpg', '450322671_372826492490389_1193008324291304709_n.jpg', '450506925_2689081387928093_168584447421921865_n.jpg', '450611234_1006272027545250_2202924425935663957_n.jpg', '9.5_SnowDepthVariationSpatialvsTemporal.pptx', 'Academic Transcript.pdf', 'Anaconda3-2024.10-1-Windows-x86_64.exe', 'archive (2)', 'archive (2).zip', 'archive.zip', 'arduino-1.8.13-windows', 'assignment3_A69036934.py', 'baseline _0.77466.ipynb', 'baseline.ipynb', 'baseline_0.79594.ipynb', 'C1.PNG', 'C2.PNG', 'C3.PNG', 'Cadence', 'Capture.PNG', 'CharArray1.1', 'chinook_postgres.sql', 'chromeremotedesktophost.msi', 'collinearPoints.ipynb', 'combined.txt', 'cuda_12.8.1_572.61_windows.exe', 'Current Resume 03032021.pdf', 'cwrsync_6.4.2_x64_free.zip', 'data', 'd

In [14]:
import os

java_home = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.27.6-hotspot"
os.environ["JAVA_HOME"] = java_home

# Get current PATH and remove ALL Java entries
path_entries = os.environ["PATH"].split(";")
cleaned_path = [p for p in path_entries if "jdk" not in p.lower() and "java" not in p.lower()]

# Add ONLY the correct Java 11 bin directory
java11_bin = os.path.join(java_home, "bin")
cleaned_path.insert(0, java11_bin)

# Set the cleaned PATH
os.environ["PATH"] = ";".join(cleaned_path)

print("CLEANED PATH:", os.environ["PATH"])

# Now import PySpark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Test") \
    .master("local[*]") \
    .getOrCreate()

file_path = "Test_data.csv"
df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)
df.show()
print("Number of partitions:", df.rdd.getNumPartitions())

CLEANED PATH: C:\Program Files\Eclipse Adoptium\jdk-11.0.27.6-hotspot\bin


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [7]:
total_classes = df.select("ClassId").distinct().count()

total_rows = df.count()

# Print the results
print(f"Total number of classes: {total_classes}")
print(f"Total number of rows in the test data: {total_rows}")

AttributeError: 'NoneType' object has no attribute 'select'

In [ ]:
image_count_per_class = df.groupBy("ClassId").count().orderBy("ClassId")

# Show the image count per class in ascending order
image_count_per_class.show(205)

In [ ]:
window_spec = Window.partitionBy("ClassId").orderBy("Path") 

# Add a row number to each image within its class
df_with_row_num = df.withColumn("row_num", row_number().over(window_spec))

# Filter to keep only the first 20 rows per class
df_20_images_per_class = df_with_row_num.filter(col("row_num") <= 20)

# Group by ClassId and count the number of images per class
image_count_per_class = df_20_images_per_class.groupBy("ClassId").count()

# Show the ClassId with the count of exactly 20 images per class
image_count_per_class.show(205)

In [ ]:
train_root = "/path/to/Train"
test_root = "/path/to/Test"
os.path.join(path, "Data_images", "Train")
os.path.join(path, "Data_images", "Test")

In [ ]:
train_root = os.path.join(
    path, 
    "Data_images",
    "Train"
)

test_root = os.path.join(
    path, 
    "Data_images",
    "Test"
)

print("Final train path:", train_root)
print("Final test path:", test_root)

In [ ]:
data = []

# Iterating over each subdirectory in "Train" (each subdir is a different class)
for subdir in sorted(os.listdir(train_root)):
    class_dir = os.path.join(train_root, subdir)
    if os.path.isdir(class_dir):
        class_id = subdir  # e.g. "0", "1", "10", ...
        
        # Iterate over all files in that subdirectory
        for filename in os.listdir(class_dir):
            full_path = os.path.join(class_dir, filename)
            if os.path.isfile(full_path):
                data.append((full_path, class_id))

# Defining a simple schema with 2 columns: Path, ClassId
schema = StructType([
    StructField("Path", StringType(), True),
    StructField("ClassId", StringType(), True),
])

# Creating a Spark DataFrame from the list of tuples
df_train = spark.createDataFrame(data, schema)

# showing a few rows
df_train.show(20, truncate=False)
print("Total rows (images) =", df_train.count())
print("Distinct classes =", df_train.select("ClassId").distinct().count())

In [ ]:
# 1. Counting total classes
total_classes_train = df_train.select("ClassId").distinct().count()
print(f"Total number of classes (Train): {total_classes_train}")

# 2. Counting total rows (images)
total_rows_train = df_train.count()
print(f"Total number of rows (Train): {total_rows_train}")

# 3. Showing image count per class in ascending order
df_train.groupBy("ClassId").count().orderBy("ClassId").show(205)

# 4. Defining a window to order images by Path within each class
window_spec_train = Window.partitionBy("ClassId").orderBy("Path")

# 5. Adding a row_number column for each image within its class
df_with_row_num_train = df_train.withColumn(
    "row_num", 
    row_number().over(window_spec_train)
)

# 6. Keeping only the first 20 images per class
df_150_images_per_class_train = df_with_row_num_train.filter(
    col("row_num") <= 150
)

# 7. Verifying that each class has exactly 150 images
df_150_images_per_class_train.groupBy("ClassId").count().orderBy("ClassId").show(205)

In [ ]:
data_test = []

# Iterate over all files in test_root
for filename in os.listdir(test_root):
    full_path = os.path.join(test_root, filename)
    if os.path.isfile(full_path):
        data_test.append((full_path, None))

# Define the same schema as before (Path, ClassId)
schema_test = StructType([
    StructField("Path", StringType(), True),
    StructField("ClassId", StringType(), True),
])

# Create a Spark DataFrame for the test data
df_test = spark.createDataFrame(data_test, schema_test)

# Showing a few rows
df_test.show(20, truncate=False)
print("Total rows (images) in test =", df_test.count())

In [ ]:
# 1. Counting total classes
total_classes_test = df_test.select("ClassId").distinct().count()
print(f"Total number of classes (Test): {total_classes_test}")

# 2. Counting total rows (images)
total_classes_test = df_test.count()
print(f"Total number of rows (Test): {total_classes_test}")

# 3. Showing image count per class in ascending order
df_test.groupBy("ClassId").count().orderBy("ClassId").show(205)

# 4. Defining a window to order images by Path within each class
window_spec_test = Window.partitionBy("ClassId").orderBy("Path")

# 5. Adding a row_number column for each image within its class
df_with_row_num_test = df_test.withColumn(
    "row_num", 
    row_number().over(window_spec_test)
)

# 6. Keeping only the first 20 images per class
df_20_images_per_class_test = df_with_row_num_test.filter(
    col("row_num") <= 20
)

# 7. Verifying that each class has exactly 20 images
df_20_images_per_class_test.groupBy("ClassId").count().orderBy("ClassId").show(205)

In [ ]:
# 1. Randomly sample a small fraction of your df_train (or df_150_images_per_class_train)
sample_df = df_train.orderBy(rand()).limit(20)  # or .sample(fraction=0.01, seed=42)
sample_rows = sample_df.collect()

# 2. For each sampled image, open and check size
sizes = set()
for row in sample_rows:
    path = row["Path"]
    with Image.open(path) as img:
        width, height = img.size
        sizes.add((width, height))

print("20 random sizes found in Train sample:", sizes)
if len(sizes) == 1:
    print("All sampled images have the same size:", sizes.pop())
else:
    print("Sampled images have varying sizes:", sizes)

In [ ]:
# 1. Randomly sample a small fraction of your df_train (or df_150_images_per_class_train)
sample_test_df = df_test.orderBy(rand()).limit(20)  # or .sample(fraction=0.01, seed=42)
sample_rows_test = sample_test_df.collect()

# 2. For each sampled image, open and check size
sizes = set()
for row in sample_rows_test:
    path = row["Path"]
    with Image.open(path) as img:
        width, height = img.size
        sizes.add((width, height))

print("20 random sizes found in Test sample:", sizes)
if len(sizes) == 1:
    print("All sampled images have the same size:", sizes.pop())
else:
    print("Sampled images have varying sizes:", sizes)

In [ ]:
# Looking at the output of a sample of our train and test datasets, we see that our random sample of 50 train and test images reveals many different (width, height) dimensions.
# This shows that the images in our dataset are not uniform in size, which we will have to do some pre-processing efforts to make them uniform / same sizes. 

In [ ]:
# Defining a Python function that returns (width, height)
def get_image_size(path):
    with Image.open(path) as img:
        return (img.width, img.height)

# Defining a Spark UDF returning a struct of (width, height)
schema_image_size = StructType([
    StructField("width", IntegerType(), True),
    StructField("height", IntegerType(), True),
])

valid_image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

df_test_filtered = df_test.filter(
    col("Path").rlike(r"\.(jpg|jpeg|png|bmp|gif)$")  # Regex to match valid image extensions
)

udf_get_image_size = udf(get_image_size, schema_image_size)

# Applying the UDF
df_with_size = df_train.withColumn("size", udf_get_image_size(col("Path")))

df_with_size1 = df_test_filtered.withColumn("size", udf_get_image_size(col("Path")))

# Grouping by or distinct on size
df_with_size.groupBy("size").count().show()

df_with_size1.groupBy("size").count().show()

# If we want columns width, height separately:
df_final = df_with_size.select(
    col("Path"),
    col("ClassId"),
    col("size.width").alias("width"),
    col("size.height").alias("height")
)
df_final.show()

df_final1 = df_with_size1.select(
    col("Path"),
    col("ClassId"),
    col("size.width").alias("width"),
    col("size.height").alias("height")
)
df_final1.show()

# Now we see how many unique (width, height) combos exist:
unique_sizes = df_final.select("width", "height").distinct().collect()
unique_sizes1 = df_final1.select("width", "height").distinct().collect()

print("First 50 unique Train image sizes:")
for size in unique_sizes[:50]:
    print(size)


print("\nFirst 50 unique Test image sizes:")
for size1 in unique_sizes[:50]:
    print(size1)

In [ ]:
train_freq = df_train.groupBy("ClassId").count().orderBy("ClassId")
train_freq_pd = train_freq.toPandas()
train_freq_pd["ClassId"] = train_freq_pd["ClassId"].astype(str)

plt.figure(figsize=(20, 6))
bars = plt.bar(train_freq_pd["ClassId"], train_freq_pd["count"], color='skyblue', edgecolor='black')

for bar in bars:
    height = bar.get_height()
    if height > 20:  
        plt.text(bar.get_x() + bar.get_width()/2, height, str(height), ha='center', va='bottom', fontsize=7)

plt.title("Histogram of Class ID Frequency in Train Data", fontsize=14)
plt.xlabel("Class ID", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
df_test_casted = df.withColumn("ClassId", df["ClassId"].cast("string"))

test_freq = df_test_casted.groupBy("ClassId").count().orderBy("ClassId")
test_freq_pd = test_freq.toPandas()
test_freq_pd["ClassId"] = test_freq_pd["ClassId"].astype(str)

plt.figure(figsize=(20, 6))
bars = plt.bar(test_freq_pd["ClassId"], test_freq_pd["count"], color='salmon', edgecolor='black')

for bar in bars:
    height = bar.get_height()
    if height > 20:
        plt.text(bar.get_x() + bar.get_width()/2, height, str(height), ha='center', va='bottom', fontsize=7)

plt.title("Histogram of Class ID Frequency in Test Data", fontsize=14)
plt.xlabel("Class ID", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
df_train_casted = df_train.withColumn("ClassId", df_train["ClassId"].cast("string"))
df_test_casted = df.withColumn("ClassId", df["ClassId"].cast("string"))
df_all = df_train_casted.select("ClassId").union(df_test_casted.select("ClassId"))

all_freq = df_all.groupBy("ClassId").count().orderBy("ClassId")
all_freq_pd = all_freq.toPandas()
all_freq_pd["ClassId"] = all_freq_pd["ClassId"].astype(str)

plt.figure(figsize=(20, 6))
bars = plt.bar(all_freq_pd["ClassId"], all_freq_pd["count"], color='mediumseagreen', edgecolor='black')

for bar in bars:
    height = bar.get_height()
    if height > 20:
        plt.text(bar.get_x() + bar.get_width()/2, height, str(height), ha='center', va='bottom', fontsize=7)

plt.title("Histogram of Class ID Frequency (Train + Test)", fontsize=14)
plt.xlabel("Class ID", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
data_images_dir = os.path.dirname(os.path.dirname(path))  

split_folders = [f for f in os.listdir(data_images_dir) if os.path.isdir(os.path.join(data_images_dir, f))]

class_images = defaultdict(list)
for split in split_folders:
    split_dir = os.path.join(data_images_dir, split)
    for class_name in os.listdir(split_dir):
        class_dir = os.path.join(split_dir, class_name)
        if os.path.isdir(class_dir):
            images = glob(os.path.join(class_dir, '*'))
            class_images[class_name].extend(images)

class_counts = {cls: len(imgs) for cls, imgs in class_images.items()}

top_classes = [cls for cls, _ in Counter(class_counts).most_common(5)]

fig, axes = plt.subplots(len(top_classes), 3, figsize=(12, 10))
for i, class_name in enumerate(top_classes):
    images = class_images[class_name]
    selected_images = random.sample(images, 3)
    for j, img_path in enumerate(selected_images):
        img = plt.imread(img_path)
        axes[i, j].imshow(img)
        axes[i, j].set_title(class_name)
        axes[i, j].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import random_split

# Define the neural network architecture
class TrafficSignNet(nn.Module):
    def __init__(self, num_classes=205):
        super(TrafficSignNet, self).__init__()
        self.conv_layers = nn.Sequential(
            # First convolutional block
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # Second convolutional block
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # Third convolutional block
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # Calculate the size of the flattened features
        self.flat_features = 128 * 4 * 4  # Adjust based on your input size
        
        self.fc_layers = nn.Sequential(
            nn.Linear(self.flat_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc_layers(x)
        return x

# Custom Dataset class
class TrafficSignDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform or transforms.Compose([
            transforms.Resize((32, 32)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df[idx]
        
        # Safely access fields
        if isinstance(row, dict):
            image_path = row["Path"]
            label = row["ClassId"]
        else:  # PySpark Row
            image_path = row.Path
            label = row.ClassId
    
        if label is None:
            raise ValueError(f"Label is None for image at path: {image_path}")
    
        label = int(label)
    
        # Load and preprocess image
        image = Image.open(image_path).convert('L')  # Convert to grayscale
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Prepare data
print("Preparing training data...")
sample_df = df_train.orderBy(rand())#.limit(5000)
sample_rows = sample_df.collect()

# Create dataset and dataloader
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

dataset = TrafficSignDataset(sample_rows, transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Initialize model, loss function, and optimizer
model = TrafficSignNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Add a StepLR scheduler: reduce LR by gamma every step_size epochs
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# Set train/val split ratio
train_ratio = 0.8
total_size = len(sample_rows)
train_size = int(train_ratio * total_size)
val_size = total_size - train_size

# Split dataset
full_dataset = TrafficSignDataset(sample_rows, transform)

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Training loop
num_epochs = 10
print("\nStarting training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    # Scheduler step
    scheduler.step()

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_acc = 100 * val_correct / val_total

    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Loss: {running_loss / len(train_loader):.4f}, '
          f'Train Acc: {100 * correct / total:.2f}%, '
          f'Val Acc: {val_acc:.2f}%')

# Evaluation on validation set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Print classification report
print("\nValidation Classification Report:")
print(classification_report(all_labels, all_preds))

# Print confusion matrix
print("\nValidation Confusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
print(cm)

# Evaluation on test set
df_test = df_test.filter((df_test.Path.isNotNull()) & (df_test.ClassId.isNotNull()))
test_rows = df_test.collect()

# Optional sanity check
print(f"Collected {len(test_rows)} test rows")

# Create dataset and loader
test_dataset = TrafficSignDataset(test_rows, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

test_image_paths = [row.Path for row in df_test.select("Path").collect()]

# Then proceed with your prediction code
results = []
model.eval()

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        results.extend(predicted.cpu().tolist())

# Zip file paths with predictions
predictions = list(zip(test_image_paths, results))

# Optional: preview
for path, pred in predictions[:10]:
    print(f"Image: {path}, Predicted class: {pred}")